In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/milamole/synesthesia-data/synthetic_synesthesia_audio.csv


In [7]:
from scipy.special import softmax

# Implementation of Logistic Regression Using Numpy

I used numpy to build the logisitic regression model without sklearn with the exception of using SciPy to use the softmax function. 

With this implementation, I learnt the importance of one-hot encoding in multi-class classification. My prior understanding of one-hot encoding was to translate categorical data into numerical data. Although that is true for one-hot encoding, I also learnt that it sometimes returns a n x n matrix that encodes a positional 1 in a row/ column of zeros in the position of that class. This helps map the text into a numeric ecosystem that the model can map much better than my original idea to use a mapping function to turn each class into values from 0 to 4.

In [8]:
class LogisticRegression:
    def __init__(self, lr= 0.001, n_iterations= 1000):
        self.lr = lr
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.classes = None
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        y_encoded = np.eye(n_classes)[y.astype(int)]
        
        self.weights = np.zeros((n_features, n_classes))
        self.bias = np.zeros(n_classes)
        
        for _ in range(self.n_iterations):
            linear_pred = np.dot(X, self.weights) + self.bias
            y_preds = softmax(linear_pred, axis =1)
            dw = 1/n_samples  * np.dot(X.T, (y_preds -y_encoded))
            db = 1/n_samples  * np.sum((y_preds -y_encoded), axis =0)
            
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
    def predict(self, X):
        logits = np.dot(X, self.weights) + self.bias
        preds = softmax(logits, axis =1)
        return np.argmax(preds, axis=1)

In [9]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("/kaggle/input/datasets/milamole/synesthesia-data/synthetic_synesthesia_audio.csv")

In [35]:
print(df["visual_state"].unique())


['warm_soft' 'sharp_chaotic' 'deep_calm' 'airy_ethereal'
 'bright_energetic']


In [36]:
print(df["visual_state"].values)


['warm_soft' 'sharp_chaotic' 'deep_calm' ... 'airy_ethereal'
 'airy_ethereal' 'deep_calm']


In [10]:
class_maps = {"warm_soft": 0, "sharp_chaotic": 1, "deep_calm": 2, "airy_ethereal": 3, "bright_energetic": 4}

In [11]:
df["visual_state"] = df["visual_state"].map(class_maps)

In [12]:
df["visual_state"][:50]

0     0
1     1
2     2
3     0
4     3
5     2
6     2
7     2
8     0
9     2
10    3
11    4
12    2
13    2
14    4
15    4
16    0
17    3
18    2
19    2
20    2
21    3
22    1
23    4
24    3
25    1
26    0
27    3
28    4
29    1
30    3
31    3
32    1
33    0
34    0
35    2
36    1
37    4
38    3
39    4
40    4
41    2
42    2
43    4
44    3
45    1
46    3
47    1
48    0
49    2
Name: visual_state, dtype: int64

In [13]:
X = df[["dominant_frequency_hz", "high_band_energy", "spectral_centroid_hz", "spectral_spread_hz", "low_band_energy", "amplitude", "mid_band_energy"]]
y = df.pop("visual_state")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size =0.3, random_state = 50)


# The Importance of Standardising Data

With a previous implementation of Logistic Regression using Numpy, I accidentally included an arbitrary id column that created large value differences in the features causing a 35% accuracy. Logisitc Regression, from what I've learnt, is best used when features are standardised. The synthetic data I used had data with values ranging from 0.n -> n * 100, so it was important that I standardised this data for the best results of this implementation of Linear Regression.

In [15]:
mean = X_train.mean(axis =0)
std = X_train.std(axis=0)

X_train_scaled= (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

In [41]:
print(y_train.shape)
print(X_train_scaled.shape)

(4200,)
(4200, 7)


In [42]:
X_test_scaled.shape

(1800, 7)

In [16]:
model = LogisticRegression()

model.fit(X_train_scaled, y_train)

In [38]:
X_test_scaled.shape

(1800, 4)

In [17]:
preds = model.predict(X_test_scaled)

In [18]:
print(type(preds))

<class 'numpy.ndarray'>


In [19]:
print(preds[:50])

[2 1 2 1 2 3 2 0 1 3 1 2 3 3 4 2 3 2 1 2 2 4 2 1 3 4 2 3 1 4 1 1 1 0 4 2 4
 1 1 0 0 2 2 2 4 2 2 3 4 3]


In [47]:
print(preds == y_test)

5129     True
294     False
460      True
4373     True
848      True
        ...  
5540     True
4864     True
2488     True
4122    False
745      True
Name: visual_state, Length: 1800, dtype: bool


In [48]:
acc = np.mean(preds == y_test) * 100

print(acc)

78.16666666666666


In [61]:
np.savez(
    "model.npz",
    weights = model.weights,
    bias= model.bias,
    classes= model.classes
)